# PLS Preview

the 9 grid circuits on 8 pls components, next to logistic regression and the sin/cos twin.

In [1]:
!pip install -q pennylane==0.45.1

import os
os.environ["OMP_NUM_THREADS"] = "1"
import time
import pickle
import multiprocessing as mp
import numpy as np
import pandas as pd
import torch
import pennylane as qml
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

seed = 0
np.random.seed(seed)
torch.set_num_threads(1)

prep_dir = "/kaggle/input/datasets/malakkhaled202300369/nic-feature-dataset/prep"
clip_dir = "/kaggle/input/datasets/malakkhaled202300369/nic-clip-dataset/features"
out_dir = "/kaggle/working/pls_preview"
os.makedirs(out_dir, exist_ok=True)

seeds = [0, 1]
lr = 0.01
epochs = 5
batch_size = 256
c_grid = [0.01, 0.1, 1.0]

In [2]:
angles = {}
meta = {}
for split in ["train", "val"]:
    angles[split] = np.load(f"{prep_dir}/ffpp_{split}_angles32.npy").astype(np.float64)
    meta[split] = pd.read_csv(f"{prep_dir}/ffpp_{split}_rows.csv")

is_inner = np.load(f"{prep_dir}/ffpp_train_is_inner.npy").astype(bool)
comp = pd.read_csv(f"{prep_dir}/per_component_auc.csv")
best8 = sorted(comp.sort_values("strength_search_train", ascending=False)["component"].head(8).tolist())
first8 = list(range(8))

y_train = meta["train"]["label"].to_numpy()
y_val = meta["val"]["label"].to_numpy()
# a video is (method, video_id), the 4 fakes of one pair share a video_id
val_video = meta["val"].groupby(["method", "video_id"]).ngroup().to_numpy()

In [3]:
clip = {}
for split in ["train", "val"]:
    clip_rows = pd.read_csv(f"{clip_dir}/ffpp_{split}_rows.csv")
    assert (clip_rows["path"].to_numpy() == meta[split]["path"].to_numpy()).all(), split
    x = np.load(f"{clip_dir}/ffpp_{split}_clip.npy").astype(np.float64)
    clip[split] = x / np.linalg.norm(x, axis=1, keepdims=True)
st = ~is_inner

pls = PLSRegression(n_components=8, scale=False)
pls.fit(clip["train"][st], y_train[st])
z = {split: pls.transform(clip[split]) for split in clip}
lo = np.percentile(z["train"][st], 0.5, axis=0)
hi = np.percentile(z["train"][st], 99.5, axis=0)
pls_angles = {split: np.clip((z[split] - lo) / (hi - lo), 0.0, 1.0) * np.pi for split in z}

for split in pls_angles:
    np.save(f"{out_dir}/ffpp_{split}_pls8_angles.npy", pls_angles[split].astype(np.float32))
with open(f"{out_dir}/pls8.pkl", "wb") as f:
    pickle.dump({"pls": pls, "angle_lo": lo, "angle_hi": hi}, f)

## the circuit from a genome

a genome is the 8 components (to qubits 0 to 7 in ascending order), the encoding, and per layer a rotation and an entangler. readout is <z> on all 8 qubits, then a weighted sum and a bias.

In [4]:
n_qubits = 8
gates_per_layer = {"none": 0, "chain": n_qubits - 1, "ring": n_qubits, "cz_ring": n_qubits}
angles_per_rot = {"RY": 1, "RX+RY": 2, "Rot": 3}


def make_genome(mask, encoding, layers):
    return {"mask": sorted(mask), "encoding": encoding, "layers": [{"rotation": r, "entangler": e} for r, e in layers]}


def two_qubit_gates(g):
    return sum(gates_per_layer[layer["entangler"]] for layer in g["layers"])


def n_weights(g):
    return sum(n_qubits * angles_per_rot[layer["rotation"]] for layer in g["layers"]) + n_qubits + 1


def build_circuit(g):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(x, ws):
        for i in range(n_qubits):
            if g["encoding"] == "RX":
                qml.RX(x[:, i], wires=i)
            elif g["encoding"] == "RY":
                qml.RY(x[:, i], wires=i)
            else:
                qml.Hadamard(wires=i)
                qml.RZ(x[:, i], wires=i)
        for layer, w in zip(g["layers"], ws):
            for i in range(n_qubits):
                if layer["rotation"] == "RY":
                    qml.RY(w[i, 0], wires=i)
                elif layer["rotation"] == "RX+RY":
                    qml.RX(w[i, 0], wires=i)
                    qml.RY(w[i, 1], wires=i)
                else:
                    qml.Rot(w[i, 0], w[i, 1], w[i, 2], wires=i)
            if layer["entangler"] == "chain":
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
            elif layer["entangler"] == "ring":
                for i in range(n_qubits):
                    qml.CNOT(wires=[i, (i + 1) % n_qubits])
            elif layer["entangler"] == "cz_ring":
                for i in range(n_qubits):
                    qml.CZ(wires=[i, (i + 1) % n_qubits])
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    return circuit


class QuantumClassifier(torch.nn.Module):
    def __init__(self, g):
        super().__init__()
        self.circuit = build_circuit(g)
        self.rot = torch.nn.ParameterList()
        for layer in g["layers"]:
            k = angles_per_rot[layer["rotation"]]
            self.rot.append(torch.nn.Parameter(0.1 * torch.randn(n_qubits, k, dtype=torch.float64)))
        self.readout = torch.nn.Linear(n_qubits, 1, dtype=torch.float64)
        torch.nn.init.normal_(self.readout.weight, std=0.1)
        torch.nn.init.zeros_(self.readout.bias)

    def forward(self, x):
        out = torch.stack(self.circuit(x, list(self.rot)), dim=1).to(torch.float64)
        return self.readout(out).squeeze(1)


def train(model, x, y, lr=0.01, epochs=5, batch_size=256, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.as_tensor(x, dtype=torch.float64)
    y = torch.as_tensor(y, dtype=torch.float64)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    for epoch in range(epochs):
        order = torch.randperm(len(x), generator=g)
        for start in range(0, len(x), batch_size):
            idx = order[start:start + batch_size]
            opt.zero_grad()
            loss_fn(model(x[idx]), y[idx]).backward()
            opt.step()


def predict(model, x, batch_size=4096):
    out = []
    with torch.no_grad():
        for s in range(0, len(x), batch_size):
            xb = torch.as_tensor(x[s:s + batch_size], dtype=torch.float64)
            out.append(torch.sigmoid(model(xb)))
    return torch.cat(out).numpy()

In [5]:
def video_level(prob):
    df = pd.DataFrame({"video": val_video, "p": prob, "y": y_val})
    g = df.groupby("video").agg(p=("p", "mean"), y=("y", "first"))
    return g["p"].to_numpy(), g["y"].to_numpy()


def bootstrap_ci(p, y, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    aucs = []
    while len(aucs) < n:
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        aucs.append(roc_auc_score(y[idx], p[idx]))
    return np.percentile(aucs, [2.5, 97.5])


def score(prob):
    pv, yv = video_level(prob)
    lo, hi = bootstrap_ci(pv, yv)
    return {"frame auc": roc_auc_score(y_val, prob), "video auc": roc_auc_score(yv, pv), "ci low": lo, "ci high": hi}

## classical references on the 8 pls angles

In [6]:
def sincos(a):
    return np.concatenate([np.sin(a), np.cos(a)], axis=1)


classical = []
for name, xt, xv in [("logistic regression", pls_angles["train"], pls_angles["val"]),
                     ("sin/cos twin", sincos(pls_angles["train"]), sincos(pls_angles["val"]))]:
    best = None
    for c in c_grid:
        model = make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=5000)).fit(xt, y_train)
        row = score(model.predict_proba(xv)[:, 1])
        if best is None or row["video auc"] > best["video auc"]:
            best = dict(row, model=name, setting=c)
    classical.append(best)

classical = pd.DataFrame(classical)
classical.to_csv(f"{out_dir}/classical_pls8.csv", index=False)
print(classical.round(3).to_string(index=False))

 frame auc  video auc  ci low  ci high               model  setting
     0.815      0.840   0.808    0.871 logistic regression      0.1
     0.816      0.842   0.809    0.872        sin/cos twin      0.1


## the 9 grid circuits on pls 8

In [7]:
grid = []
for depth in [1, 2, 3]:
    for ent in ["none", "chain", "ring"]:
        g = make_genome(first8, "RY", [("Rot", ent)] * depth)
        grid.append({"circuit": f"pls 8, depth {depth}, {ent}", "genome": g, "weights": n_weights(g), "gates": two_qubit_gates(g)})


def run_task(task):
    i, s = task
    t0 = time.time()
    torch.manual_seed(s)
    model = QuantumClassifier(grid[i]["genome"])
    train(model, pls_angles["train"], y_train, lr=lr, epochs=epochs, batch_size=batch_size, seed=s)
    row = {"i": i, "seed": s}
    row.update(score(predict(model, pls_angles["val"])))
    row["seconds"] = time.time() - t0
    return row


tasks = []
for i in range(len(grid)):
    for s in seeds:
        tasks.append((i, s))
with mp.get_context("fork").Pool(os.cpu_count()) as pool:
    results = pool.map(run_task, tasks)

In [8]:
runs = pd.DataFrame(results)
runs["circuit"] = [grid[i]["circuit"] for i in runs["i"]]
runs.to_csv(f"{out_dir}/pls_grid_runs.csv", index=False)

summary = []
for i, c in enumerate(grid):
    f = runs[runs["i"] == i]
    summary.append({"circuit": c["circuit"], "weights": c["weights"], "gates": c["gates"],
                    "video auc mean": f["video auc"].mean(), "video auc std": f["video auc"].std(),
                    "ci low worst": f["ci low"].min(), "ci high best": f["ci high"].max()})
summary = pd.DataFrame(summary)
summary.to_csv(f"{out_dir}/pls_grid_summary.csv", index=False)
print(summary.round(3).to_string(index=False))

              circuit  weights  gates  video auc mean  video auc std  ci low worst  ci high best
 pls 8, depth 1, none       33      0           0.841          0.000         0.808         0.872
pls 8, depth 1, chain       33      7           0.799          0.024         0.744         0.851
 pls 8, depth 1, ring       33      8           0.798          0.021         0.745         0.847
 pls 8, depth 2, none       57      0           0.840          0.000         0.808         0.872
pls 8, depth 2, chain       57     14           0.843          0.004         0.806         0.875
 pls 8, depth 2, ring       57     16           0.841          0.005         0.804         0.873
 pls 8, depth 3, none       81      0           0.841          0.000         0.808         0.872
pls 8, depth 3, chain       81     21           0.839          0.003         0.805         0.871
 pls 8, depth 3, ring       81     24           0.841          0.000         0.809         0.871
